In [ ]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import csv
import nltk
from nltk.tokenize import sent_tokenize
import optuna
from optuna.pruners import MedianPruner

Create META dataframe to save corresponding meta data and break articles into sentences for SBERT. 

In [ ]:
from nltk.tokenize import sent_tokenize
data = pd.read_csv("")

# --- split into sentences, keeping article metadata aligned ---
sent_lists = [sent_tokenize(doc) for doc in data["text_clean"].fillna("")]
sentences  = [s for doc in sent_lists for s in doc]

counts = [len(d) for d in sent_lists]
meta = data.loc[data.index.repeat(counts)].reset_index(drop=True)
meta["sentence"] = sentences          # 1:1 aligned with `sentences`

# drop empties / very short fragments
mask = meta["sentence"].str.strip().str.len() > 0
meta = meta[mask].reset_index(drop=True)
sentences = meta["sentence"].tolist()

Set up the Model

In [ ]:
#Updated with Optuna Parameters

vectorizer_model_all = CountVectorizer(ngram_range=(1, 3), stop_words="english", min_df=10)

# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(sentences, show_progress_bar=True)

from umap import UMAP 
umap_model = UMAP(n_neighbors=40, n_components=19, min_dist=0.0, metric='cosine', random_state=42)

from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=150, min_samples=25, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

#representation_model = MaximalMarginalRelevance(diversity=0.5) #previously 0.3 

In [ ]:
import openai
import tiktoken
from bertopic.representation import OpenAI
from bertopic import BERTopic

# Tokenizer
tokenizer= tiktoken.encoding_for_model("gpt-4")


client = openai.OpenAI(api_key="")
representation_model = OpenAI(
    client,
    model="gpt-4",
    delay_in_seconds=2,
    chat=True,
    nr_docs=15,
    doc_length=100,
    tokenizer=tokenizer
)

In [ ]:
topic_model = BERTopic(

  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model_all,
  representation_model = representation_model,

  # BERT's own Hyperparameters
  top_n_words=10,
  min_topic_size= 85,
  nr_topics = "auto",
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(sentences, embeddings)
# attach results back to metadata
meta["topic"] = topics

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.get_topics()

In [ ]:
topic_model.save(
    "my_topic_model",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="sentence-transformers/all-MiniLM-L6-v2"  # pass the name, not the object
)

In [ ]:
topic_model.merge_topics(sentences, [41, 55, 19])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [54, 24])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [24, 6, 28, 32])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [27, 25, 30, 19, 47, 15])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [-1, 23, 52, 29, 18, 47, 51, 17, 16])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [4, 31])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.get_topics()

topic_model.merge_topics(sentences, [27, 37, 36, 13, 16, 35, 6, 38, 23, 41, 32])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [2, 18, 21, 30,])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.merge_topics(sentences, [18, 9])

In [ ]:
topic_model.merge_topics(sentences, [6, 28])

In [ ]:
topic_model.merge_topics(sentences, [20,22,18])

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.merge_topics(sentences, [5, 9, 12])

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.merge_topics(sentences, [[19, 17],
                    [10, 2, 5, 21], [4, 6], [0,23]])

In [ ]:
topic_model.get_topics()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
assert len(topic_model.topics_) == len(meta)
meta = meta.reset_index(drop=True)  # only if meta's order matches docs' order
meta["topic"] = topic_model.topics_